# Bronze Layer Setup: Schema and Volume Creation

**Purpose:** This notebook creates the foundational data architecture for the  Retail Store analytics project.

**What This Notebook Does:**
* Creates three schemas: `bronze`, `silver`, and `gold` for the medallion architecture
* Creates a Unity Catalog volume `raw_data` in the bronze schema for manual file uploads
* Verifies the setup is ready for data ingestion

**Next Steps After Running:**
* Manually upload raw CSV/Parquet files to the bronze volume
* Run notebook 1-4 to read files and create silver layer tables

## 1. Configuration

**What We're Setting:**
* Catalog name for Unity Catalog
* Schema names for bronze, silver, and gold layers
* Volume name for raw data storage

**Why This Matters:**
* Centralizes configuration for easy updates
* Ensures consistency across all notebooks in the project
* Follows Unity Catalog naming conventions

In [0]:
# Configuration for Unity Catalog setup
catalog_name = "workspace"  # Using workspace catalog for this project
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"
raw_data_volume = "raw_data"

# Display configuration
print("Project Configuration:")
print(f"Catalog: {catalog_name}")
print(f"Bronze Schema: {catalog_name}.{bronze_schema}")
print(f"Silver Schema: {catalog_name}.{silver_schema}")
print(f"Gold Schema: {catalog_name}.{gold_schema}")
print(f"Raw Data Volume: {catalog_name}.{bronze_schema}.{raw_data_volume}")

Project Configuration:
Catalog: workspace
Bronze Schema: workspace.bronze
Silver Schema: workspace.silver
Gold Schema: workspace.gold
Raw Data Volume: workspace.bronze.raw_data


## 2. Create Schemas

**What We're Creating:**
* Bronze schema: Landing zone for raw, unprocessed data
* Silver schema: Cleaned and enriched data with business logic applied
* Gold schema: Business-ready aggregated tables for reporting and dashboards

**Why This Matters:**
* Medallion architecture separates data quality layers
* Bronze preserves raw data for audit and reprocessing
* Silver enables analytics queries with clean, validated data
* Gold provides pre-aggregated metrics for fast dashboard performance

**Expected:**
* Schemas created if they don't exist
* No error if schemas already exist (using IF NOT EXISTS)

In [0]:
%sql
-- Create bronze schema for raw data landing zone
CREATE SCHEMA IF NOT EXISTS workspace.bronze
COMMENT 'Bronze layer: Raw data files loaded from source systems. Minimal transformations, preserves original structure for audit and reprocessing.'

In [0]:
%sql
-- Create silver schema for cleaned and enriched data
CREATE SCHEMA IF NOT EXISTS workspace.silver
COMMENT 'Silver layer: Cleaned, validated, and enriched data. Business rules applied, data quality checks passed, ready for analytics.'

In [0]:
%sql
-- Create gold schema for business-ready aggregated tables
CREATE SCHEMA IF NOT EXISTS workspace.gold
COMMENT 'Gold layer: Pre-aggregated business metrics and KPIs. Optimized for dashboard queries and executive reporting.'

## 3. Create Raw Data Volume

**What We're Creating:**
* Unity Catalog managed volume in the bronze schema
* Storage location for manually uploaded raw files (CSV, Parquet, JSON)

**Why This Matters:**
* Volumes provide governed file storage within Unity Catalog
* Enables direct file access with Unity Catalog permissions
* Keeps raw files organized and traceable
* Files in volumes can be read using `read_files()` function

**Expected:**
* Volume created at path `/Volumes/workspace/bronze/raw_data/`
* Ready to accept manual file uploads via Databricks UI or CLI

In [0]:
%sql
-- Create volume for raw data file uploads
CREATE VOLUME IF NOT EXISTS workspace.bronze.raw_data
COMMENT 'Storage for raw CSV/Parquet files uploaded from source systems. Upload files here before running silver layer ingestion.'

## 4. Verify Setup

**What We're Checking:**
* All three schemas exist and are accessible
* Raw data volume is created and ready
* Volume path is correct for file uploads

**Why This Matters:**
* Confirms infrastructure is ready for data ingestion
* Provides volume path for manual file uploads
* Validates permissions are correct

In [0]:
%sql
-- Verify all schemas were created
SHOW SCHEMAS IN workspace

databaseName
bronze
default
gold
information_schema
silver


In [0]:
%sql
-- Verify volume was created in bronze schema
SHOW VOLUMES IN workspace.bronze

database,volume_name
bronze,raw_data


In [0]:
# Display the volume path for file uploads
volume_path = f"/Volumes/{catalog_name}/{bronze_schema}/{raw_data_volume}"

print("✓ Setup Complete!")
print("\n" + "="*60)
print("VOLUME PATH FOR FILE UPLOADS:")
print("="*60)
print(f"{volume_path}")

✓ Setup Complete!

VOLUME PATH FOR FILE UPLOADS:
/Volumes/workspace/bronze/raw_data


Next Steps:
1. Upload your raw CSV/Parquet files to this volume
2. Run notebook 1-4 to ingest files and create silver layer tables

To upload files:
  - Via UI: Navigate to Data > Volumes > workspace > bronze > raw_data
  - Via CLI: databricks fs cp local_file.csv dbfs:/Volumes/workspace/bronze/raw_data/


## 5. Summary

**What Was Created:**
* Three Unity Catalog schemas implementing medallion architecture:
  * `workspace.bronze` - Raw data landing zone
  * `workspace.silver` - Cleaned and enriched analytics tables
  * `workspace.gold` - Pre-aggregated business metrics
* Volume `workspace.bronze.raw_data` - Managed storage for raw file uploads

**Key Concepts Applied:**
* **Medallion Architecture:** Progressive data refinement from raw to business-ready
* **Unity Catalog Volumes:** Governed file storage with UC permissions
* **IF NOT EXISTS:** Idempotent SQL for safe re-execution
* **Schema Comments:** Documentation embedded in catalog metadata

**Architecture Flow:**
```
Raw Files → Bronze Volume → Silver Tables → Gold Summary Tables
  (CSV)      (UC Volume)    (Cleaned Data)   (Aggregated KPIs)
```

**This Notebook's Role:**
* One-time infrastructure setup
* Creates the foundation for data ingestion
* No data processing - just schema and volume creation
* Can be re-run safely without errors